In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
TRAIN_RATIO = 0.7
VAL_RATIO = 0.1
TEST_RATIO = 0.2
RANDOM_SEED = 123

In [3]:
rng = np.random.default_rng(RANDOM_SEED)

In [4]:
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-6

In [5]:
df = pd.read_csv("HAM10000_metadata_official.csv")

In [6]:
df.drop(columns=["dx_type", "dataset"], inplace=True)

In [7]:
df["dataset"] = "ham10000"

In [8]:
df.rename(columns={
    "image_id": "image_name",
    "dx": "label",
    "lesion_id": "patient_id",
    "localization": "location",
}, inplace=True)

In [9]:
df["age"] = df["age"].fillna("unknown")

In [10]:
df.isna().sum()

patient_id    0
image_name    0
label         0
age           0
sex           0
location      0
dataset       0
dtype: int64

In [11]:
assert df.isnull().sum().sum() == 0

In [12]:
df["image_name"] = df["image_name"].apply(
    lambda x: x if str(x).endswith(".jpg") else str(x) + ".jpg")

In [13]:
assert len(df) == df["image_name"].nunique()

In [14]:
missing = []
for f in df["image_name"]:
    if not os.path.exists(os.path.join("/home/prasad/datasets/ham10000/images/", f)):
        missing.append(f)

print(len(missing))

0


In [15]:
mapping = {
    "akiec": "AKIEC",
    "bcc"  : "BCC",
    "bkl"  : "BKL",
    "df"   : "DF",
    "mel"  : "MEL",
    "nv"   : "NEV",
    "vasc" : "VASC",
}

In [16]:
assert set(df["label"].unique()) == set(mapping.keys())

In [17]:
df["label"] = df["label"].map(mapping)

In [18]:
assert df["label"].isna().sum() == 0

In [19]:
assert sorted(df["label"].unique()) == [
    "AKIEC", "BCC", "BKL", "DF", "MEL", "NEV", "VASC"]

In [20]:
df["label"].value_counts()

label
NEV      6705
MEL      1113
BKL      1099
BCC       514
AKIEC     327
VASC      142
DF        115
Name: count, dtype: int64

In [21]:
print(df["label"].value_counts().sum())

10015


In [22]:
assert df["label"].isna().sum() == 0

In [23]:
df["binary_label"]=(df["label"].isin(
    ["MEL", "BCC", "AKIEC"])).astype(int)

In [24]:
assert set(df["binary_label"].unique()) == {0,1}

In [25]:
print(df["binary_label"].value_counts())

binary_label
0    8061
1    1954
Name: count, dtype: int64


In [26]:
print(df["binary_label"].value_counts().sum())

10015


In [27]:
print(pd.crosstab(df["binary_label"], df["label"]))

label         AKIEC  BCC   BKL   DF   MEL   NEV  VASC
binary_label                                         
0                 0    0  1099  115     0  6705   142
1               327  514     0    0  1113     0     0


In [28]:
df["sex"].value_counts()

sex
male       5406
female     4552
unknown      57
Name: count, dtype: int64

In [29]:
print(df["sex"].value_counts().sum())

10015


In [30]:
print(pd.crosstab(df["sex"], df["label"]))

label    AKIEC  BCC  BKL  DF  MEL   NEV  VASC
sex                                          
female     106  197  463  52  424  3237    73
male       221  317  626  63  689  3421    69
unknown      0    0   10   0    0    47     0


In [31]:
df["age"].value_counts()

age
45.0       1299
50.0       1187
55.0       1009
40.0        985
60.0        803
70.0        756
35.0        753
65.0        731
75.0        618
30.0        464
80.0        404
85.0        290
25.0        247
20.0        169
5.0          86
15.0         77
unknown      57
10.0         41
0.0          39
Name: count, dtype: int64

In [32]:
df["age"].value_counts().sum()

np.int64(10015)

In [33]:
df["age"].isna().sum()

np.int64(0)

In [34]:
print(pd.crosstab(df["age"], df["label"]))

label    AKIEC  BCC  BKL  DF  MEL   NEV  VASC
age                                          
0.0          0    0    5   0    0    30     4
5.0          0    0    1   0    1    81     3
10.0         0    0    0   0    0    39     2
15.0         0    0    0   0    0    73     4
20.0         0    1    0   0    6   158     4
25.0         0    3    0   2   16   221     5
30.0         1    4    6   4   34   410     5
35.0         0    5   24  12   36   668     8
40.0         9   23   46   9   49   846     3
45.0        10   26   59  14   74  1100    16
50.0        19   27   87  18   96   928    12
55.0        27   25   95  13  142   686    21
60.0        58   35  131   9  106   454    10
65.0        38   79  108  18  133   351     4
70.0        56   85  183   4  166   248    14
75.0        47   76  153   9   91   231    11
80.0        37   73   98   3   85    97    11
85.0        25   52   93   0   76    39     5
unknown      0    0   10   0    2    45     0


In [35]:
df["location"].value_counts()

location
back               2192
lower extremity    2077
trunk              1404
upper extremity    1118
abdomen            1022
face                745
chest               407
foot                319
unknown             234
neck                168
scalp               128
hand                 90
ear                  56
genital              48
acral                 7
Name: count, dtype: int64

In [36]:
df["location"].value_counts().sum()

np.int64(10015)

In [37]:
df["location"].isna().sum()

np.int64(0)

In [38]:
print(pd.crosstab(df["location"], df["label"]))

label            AKIEC  BCC  BKL  DF  MEL   NEV  VASC
location                                             
abdomen              5   18   42   4   66   860    27
acral                0    0    0   0    0     7     0
back                29  186  202   2  324  1427    22
chest               12   47   74   0   68   200     6
ear                  3    0    6   0   17    30     0
face               113  101  319   0  104   100     8
foot                 0    4    1   2   28   284     0
genital              0    0    2   1    0    45     0
hand                13    2   14   0    1    55     5
lower extremity     65   58  174  82  192  1479    27
neck                10   14   31   0   29    81     3
scalp               14   19   34   0   14    45     2
trunk                1   11   73   0   47  1241    31
unknown              0    5   23   0   10   196     0
upper extremity     62   49  104  24  213   655    11


# Split

In [39]:
patient_ids = df["patient_id"].dropna().unique().to_numpy()

In [40]:
rng.shuffle(patient_ids)

In [41]:
df["patient_id"].isna().sum()

np.int64(0)

In [42]:
n_patients = len(patient_ids)
n_train = int(TRAIN_RATIO * n_patients)
n_val = int(VAL_RATIO * n_patients)

In [43]:
train_patients = patient_ids[:n_train]
val_patients = patient_ids[n_train:n_train + n_val]
test_patients = patient_ids[n_train + n_val:]

In [44]:
train_ids = set(train_patients)
val_ids   = set(val_patients)
test_ids  = set(test_patients)

print("Train ∩ Val :", train_ids & val_ids)
print("Train ∩ Test:", train_ids & test_ids)
print("Val ∩ Test  :", val_ids & test_ids)

Train ∩ Val : set()
Train ∩ Test: set()
Val ∩ Test  : set()


In [45]:
df["split"] = "unassigned"
df.loc[df["patient_id"].isin(train_patients), "split"] = "train"
df.loc[df["patient_id"].isin(val_patients), "split"] = "val"
df.loc[df["patient_id"].isin(test_patients), "split"] = "test"

In [46]:
assert (df["split"] != "unassigned").all()

In [47]:
print("Patients per split:")
print(df.groupby("split")["patient_id"].nunique())

Patients per split:
split
test     1494
train    5229
val       747
Name: patient_id, dtype: int64


In [48]:
print("\nImages per split:")
print(df["split"].value_counts())


Images per split:
split
train    7034
test     1982
val       999
Name: count, dtype: int64


In [49]:
print("\nClass distribution per split:")
print(pd.crosstab(df["split"], df["label"]))


Class distribution per split:
label  AKIEC  BCC  BKL  DF  MEL   NEV  VASC
split                                      
test      72  109  194  24  220  1342    21
train    218  357  783  82  788  4699   107
val       37   48  122   9  105   664    14


In [50]:
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))

label  AKIEC  BCC  BKL  DF  MEL   NEV  VASC
split                                      
test      72  109  194  24  220  1342    21
train    218  357  783  82  788  4699   107
val       37   48  122   9  105   664    14


In [51]:
pd.crosstab(df["age"], df["split"])

split,test,train,val
age,,,
0.0,8,31,0
5.0,17,64,5
10.0,13,24,4
15.0,11,56,10
20.0,29,115,25
25.0,52,160,35
30.0,90,325,49
35.0,139,529,85
40.0,190,704,91


In [52]:
pd.crosstab(df["sex"], df["split"])

split,test,train,val
sex,,,
female,906,3169,477
male,1066,3825,515
unknown,10,40,7


In [53]:
pd.crosstab(df["location"], df["split"])

split,test,train,val
location,,,
abdomen,216,712,94
acral,0,7,0
back,410,1571,211
chest,98,268,41
ear,8,41,7
face,143,530,72
foot,78,211,30
genital,14,32,2
hand,22,60,8


In [54]:
df.groupby(["split", "sex", "label"]).size()

split  sex      label
test   female   AKIEC      29
                BCC        50
                BKL        84
                DF          9
                MEL        83
                NEV       638
                VASC       13
       male     AKIEC      43
                BCC        59
                BKL       107
                DF         15
                MEL       137
                NEV       697
                VASC        8
       unknown  BKL         3
                NEV         7
train  female   AKIEC      61
                BCC       132
                BKL       314
                DF         40
                MEL       305
                NEV      2267
                VASC       50
       male     AKIEC     157
                BCC       225
                BKL       464
                DF         42
                MEL       483
                NEV      2397
                VASC       57
       unknown  BKL         5
                NEV        35
val    female   AK

In [55]:
assert df["image_name"].is_unique
assert df["split"].isin(["train", "val", "test"]).all()
assert df["sex"].isin(["male", "female", "unknown"]).all()
assert df["label"].notna().all()

In [56]:
leakage = (
    df.groupby("patient_id")["split"]
    .nunique()
    .gt(1)
    .sum()
)
print(f"Patients appearing in multiple splits: {leakage}")

Patients appearing in multiple splits: 0


In [57]:
df=df[["dataset", "split", "image_name", "label", "binary_label", "patient_id",
        "age", "sex", "location"]]

In [58]:
expected_cols = [
    "dataset",
    "split",
    "image_name",
    "label",
    "binary_label",
    "patient_id",
    "age",
    "sex",
    "location",
]
assert list(df.columns) == expected_cols

In [59]:
print("=" * 60)
print("HAM10000 metadata preprocessing completed")
print(f"Images   : {len(df)}")
print(f"Patients : {df['patient_id'].nunique()}")
print(f"Classes  : {df['label'].nunique()}")
print(df["split"].value_counts())
print("=" * 60)

HAM10000 metadata preprocessing completed
Images   : 10015
Patients : 7470
Classes  : 7
split
train    7034
test     1982
val       999
Name: count, dtype: int64


In [60]:
print(df.isnull().sum())

dataset         0
split           0
image_name      0
label           0
binary_label    0
patient_id      0
age             0
sex             0
location        0
dtype: int64


In [61]:
assert df["image_name"].is_unique
assert df["patient_id"].notna().all()
assert df["label"].notna().all()
assert df["binary_label"].isin([0, 1]).all()
assert df["split"].isin(["train", "val", "test"]).all()

In [62]:
assert df.isnull().sum().sum() == 0

In [63]:
df.to_csv("ham10000_metadata_clean.csv", index=False, encoding="utf-8")

In [64]:
df

,dataset,split,image_name,label,binary_label,patient_id,age,sex,location
0,ham10000,train,ISIC_0027419.jpg,BKL,0,HAM_0000118,80.0,male,scalp
1,ham10000,train,ISIC_0025030.jpg,BKL,0,HAM_0000118,80.0,male,scalp
2,ham10000,val,ISIC_0026769.jpg,BKL,0,HAM_0002730,80.0,male,scalp
3,ham10000,val,ISIC_0025661.jpg,BKL,0,HAM_0002730,80.0,male,scalp
4,ham10000,train,ISIC_0031633.jpg,BKL,0,HAM_0001466,75.0,male,ear
...,...,...,...,...,...,...,...,...,...
10010,ham10000,test,ISIC_0033084.jpg,AKIEC,1,HAM_0002867,40.0,male,abdomen
10011,ham10000,test,ISIC_0033550.jpg,AKIEC,1,HAM_0002867,40.0,male,abdomen
10012,ham10000,test,ISIC_0033536.jpg,AKIEC,1,HAM_0002867,40.0,male,abdomen
10013,ham10000,train,ISIC_0032854.jpg,AKIEC,1,HAM_0000239,80.0,male,face
